In [1]:
#Loading in Packages and Data

#Importing Packages
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as colors
import matplotlib.ticker as ticker
import matplotlib.cm as cm
from matplotlib.colors import Normalize
from matplotlib.ticker import MaxNLocator
from matplotlib.ticker import ScalarFormatter
import matplotlib.gridspec as gridspec
import xarray as xr
import os; import time
import pickle
import h5py
###############################################################
def coefs(coefficients,degree):
    coef=coefficients
    coefs=""
    for n in range(degree, -1, -1):
        string=f"({coefficients[len(coef)-(n+1)]:.1e})"
        coefs+=string + f"x^{n}"
        if n != 0:
            coefs+=" + "
    return coefs
###############################################################

# #Importing Model Data
    
# dir='/mnt/lustre/koa/koastore/torri_group/air_directory/DCI-Project/'
# netCDF=xr.open_dataset(dir+'../cm1r20.3/run/cm1out_test7tundra-7_062217.nc') #***
# true_time=netCDF['time']
# # parcel=xr.open_dataset(dir+'../cm1r20.3/run/cm1out_pdata_test5tundra-7_062217.nc') #***
# times=netCDF['time'].values/(1e9 * 60); times=times.astype(float);
# Np_str='125e3'
# #Restricts the timesteps of the data from timesteps0 to 140
# data=netCDF.isel(time=np.arange(0,140+1))
# # parcel=parcel.isel(time=np.arange(0,140+1))
# res='1km'

dir='/mnt/lustre/koa/koastore/torri_group/air_directory/DCI-Project/'
netCDF=xr.open_dataset(dir+'../cm1r20.3/run/cm1out_1km_1e6.nc') #***
true_time=netCDF['time']
# parcel=xr.open_dataset(dir+'../cm1r20.3/run/cm1out_pdata_1km_1e6.nc') #***
times=netCDF['time'].values/(1e9 * 60); times=times.astype(float);
Np_str='1e6'
#Restricts the timesteps of the data from timesteps0 to 140
data=netCDF
# data=netCDF.isel(time=np.arange(0,140+1))
# parcel=parcel.isel(time=np.arange(0,140+1))
res='1km'

# #uncomment if using 250m data
# #Importing Model Data
# check=False
# dir2='/home/air673/koa_scratch/'
# data=xr.open_dataset(dir2+'cm1out_250m.nc') #***
# # # parcel=xr.open_dataset(dir2+'cm1out_pdata_250m.nc') #***

# # Restricts the timesteps of the data from timesteps0 to 140
# data=data.isel(time=np.arange(0,400+1))
# # # parcel=parcel.isel(time=np.arange(0,400+1))
# res='250m'

In [2]:
# #250 M DATA
# ###########################################################################################

# #Loading in Packages and Data

# #Importing Packages
# import numpy as np
# import matplotlib.pyplot as plt
# import matplotlib.colors as colors
# import matplotlib.ticker as ticker
# import matplotlib.cm as cm
# from matplotlib.colors import Normalize
# from matplotlib.ticker import MaxNLocator
# from matplotlib.ticker import ScalarFormatter
# import matplotlib.gridspec as gridspec
# import xarray as xr
# import os; import time
# import pickle
# import h5py
# ###############################################################
# def coefs(coefficients,degree):
#     coef=coefficients
#     coefs=""
#     for n in range(degree, -1, -1):
#         string=f"({coefficients[len(coef)-(n+1)]:.1e})"
#         coefs+=string + f"x^{n}"
#         if n != 0:
#             coefs+=" + "
#     return coefs
# ###############################################################

# #Importing Model Data
# check=False
# dir='/home/air673/koa_scratch/'

# netCDF=xr.open_dataset(dir+'cm1out.nc') #***
# res='250m'

In [3]:
def load_vars(data):
    print('PRESSURE VARIABLES'); ################################# PRESSURE VARIABLES
    p0=1e5
    P=data['prs'].data

   # print('THERMODYNAMICS'); ################################# THERMODYNAMICS
    Rd=287.04
    # Rv=461.5
    Cpd=1005.7 #+-2.5 #****divide by this
    Cpv=1870 #+-25
    Cpl=4190 #+-30
    Lv0=2.501e6
    def Lv(T): #Kirchoff's formula L_i,ii= L_i,ii0+(Cpii-Cpi)*(T-273.15)
        Llv=Lv0+(Cpv-Cpl)*(T-273.15) #should it be Cpl. is Cl the same?***
        return Llv
    
    # print('TEMPERATURE'); ################################# TEMPERATURE
    theta=data['th'].data
    T=theta*(P/p0)**(Rd/Cpd)

    # print('Specific Humidity'); ################################# Specific Humidity
    rv=data['qv'].data
    q=rv/(1+rv)

    ################################# Geopotential Height
    Nz,Ny,Nx=len(netCDF['zh']),len(netCDF['yh']),len(netCDF['xh'])
    zh_values=data['zh'].values*1000
    Z=np.broadcast_to(zh_values[:, np.newaxis, np.newaxis], (Nz, Ny, Nx))
    g=9.81
    gZ=g*Z
    
    return Cpd,T,Lv,q,gZ

In [4]:
def make_MSE(Cpd,T,q,gZ):
    # 3e2*1e3 + 2e6*5e-3 + 10 * 1e4 #~ 310000-410000 J/kg
    MSE=Cpd*T+Lv(T)*q+gZ
    return MSE

In [5]:
dir2='/mnt/lustre/koa/koastore/torri_group/air_directory/DCI-Project/'
# dir2='/mnt/lustre/koa/scratch/air673/'
def initiate_array():
    # Define array dimensions (adjust based on your netCDF)
    t_size = len(netCDF['time'])  # Number of timesteps
    z_size = len(netCDF['zh'])    # Number of vertical levels
    y_size = len(netCDF['yh'])    # Number of y-axis points
    x_size = len(netCDF['xh'])    # Number of x-axis points

    out_file=dir2 + 'Variable_Calculation/' + 'MSE'+f'_{res}_{Np_str}_5min'+'.h5'
    with h5py.File(out_file, 'a') as f:
        # Check if the dataset 'theta_e' already exists
        if 'MSE' not in f:
            # Create a dataset with the full size for all time steps (initially empty)
            f.create_dataset('MSE', 
                             (t_size, z_size, y_size, x_size),  # Full size for all timesteps
                             maxshape=(None, z_size, y_size, x_size),  # Unlimited timesteps (can grow along time dimension)
                             dtype='float64', 
                             chunks=(1, z_size, y_size, x_size))  # Chunks for time axis to allow resizing

            
def add_timestep_at_index(timestep_data, index):
    out_file=dir2 + 'Variable_Calculation/' + 'MSE'+f'_{res}_{Np_str}_5min'+'.h5'
    with h5py.File(out_file, 'a') as f:
        # Access the existing dataset 'MSE'
        dataset = f['MSE']
        
        # Assign the new timestep data at the specified index
        dataset[index] = timestep_data

In [6]:
#RUNNING

In [7]:
#MAKING ARRAY TO STORE THETA_E
initiate_array()

#CALCULATING AND APPENDING TO DATA EACH TIMESTEP
for t in range(len(netCDF['time'])):
    if np.mod(t,1)==0: print(f'Current time {t}')
    data=netCDF.isel(time=t)
    [Cpd,T,Lv,q,gZ] = load_vars(data)
    MSE=make_MSE(Cpd,T,q,gZ)
    add_timestep_at_index(MSE, t)


#Fast for 1KM, longer for 250m

Current time 0
PRESSURE VARIABLES
Current time 1
PRESSURE VARIABLES
Current time 2
PRESSURE VARIABLES
Current time 3
PRESSURE VARIABLES
Current time 4
PRESSURE VARIABLES
Current time 5
PRESSURE VARIABLES
Current time 6
PRESSURE VARIABLES
Current time 7
PRESSURE VARIABLES
Current time 8
PRESSURE VARIABLES
Current time 9
PRESSURE VARIABLES
Current time 10
PRESSURE VARIABLES
Current time 11
PRESSURE VARIABLES
Current time 12
PRESSURE VARIABLES
Current time 13
PRESSURE VARIABLES
Current time 14
PRESSURE VARIABLES
Current time 15
PRESSURE VARIABLES
Current time 16
PRESSURE VARIABLES
Current time 17
PRESSURE VARIABLES
Current time 18
PRESSURE VARIABLES
Current time 19
PRESSURE VARIABLES
Current time 20
PRESSURE VARIABLES
Current time 21
PRESSURE VARIABLES
Current time 22
PRESSURE VARIABLES
Current time 23
PRESSURE VARIABLES
Current time 24
PRESSURE VARIABLES
Current time 25
PRESSURE VARIABLES
Current time 26
PRESSURE VARIABLES
Current time 27
PRESSURE VARIABLES
Current time 28
PRESSURE VARIA

In [8]:
# #READING FINAL OUTPUT
# dir2='/mnt/lustre/koa/koastore/torri_group/air_directory/DCI-Project/'
# # dir2='/mnt/lustre/koa/scratch/air673/'
# with h5py.File(dir2 + 'Variable_Calculation/' + 'MSE'+f'_{res}_{Np_str}'+'.h5', 'a') as f:
#     # Access the existing dataset 'MSE'
#     dataset = f['MSE'][:]

In [8]:
# plt.contourf(dataset[50,0])
# plt.colorbar(label='J/kg')
# plt.title("MSE at t = 50, z = 30 m")